In [2]:
import os
import pandas as pd
import numpy as np
import pickle
import boto3
import json
import math
import datetime as dt
from tqdm import tqdm
from io import StringIO
import warnings
warnings.filterwarnings('ignore')

#### Constants

In [9]:
# project
str_project = os.getcwd().split('/')[4].replace('_','-')
print(f'Project: {str_project}')
# task
str_task = os.getcwd().split('/')[5]
print(f'Task: {str_task}')
# sub task
str_subtask = os.getcwd().split('/')[6]
print(f'Task: {str_subtask}')
# output
str_dirname_output = './output'
int_n_requests_per_lambda = 30

Project: 20240423-gen-xii-payload-parsing
Task: 02_lambda_function
Task: 01_create_parser


#### Make output

In [3]:
try:
    os.mkdir(str_dirname_output)
except:
    pass

#### Write functions.py

In [4]:
%%writefile functions.py

import pandas as pd
from io import StringIO
import numpy as np
import datetime as dt
import xml.etree.ElementTree as ET

# helper for get data
def get_data_helper(dict_json_request):
    # get rows
    list_dict_data = dict_json_request['rows']
    # get id and tables
    list_unique_id = []
    dict_list_tables = {}
    for dict_data in list_dict_data:
        # get unique_id
        unique_id = dict_data['row_id']
        list_unique_id.append(unique_id)
        # assign tables
        dict_list_tables[unique_id] = dict_data['sources']
    # return
    return list_unique_id, dict_list_tables

# get application table
def get_application_table(str_values, unique_id, dict_list_errors):
    # convert str_values to a df
    X = pd.read_csv(
        StringIO(str_values), 
        delimiter=',', 
    )
    # if X is empty
    if X.empty:
        # append error
        dict_list_errors[unique_id].append('Missing Application Table')
        # create a single row df filled with NaN
        list_cols = [
            'app_was_empty',
        ]
        X = pd.DataFrame({col: np.nan for col in list_cols}, index=[0])
        # set date to today
        X['applicationdate'] = dt.datetime.today()
    else:
        # lower
        X.columns = [col.lower() for col in X.columns]
    # add suffix to end of column names
    X.columns = [f'{col}__app' for col in X.columns]
    # set index to unique id
    X.index = [unique_id]
    # return
    return X, dict_list_errors

# get income table
def get_income_table(str_values, unique_id, dict_list_errors):
    # list of columns for income
    list_cols = [
        'bitinvalid',
        'bituse',
        'fltgrossmonthly',
    ]
    # list of cols after aggregation
    list_cols_agg = [
        'fltgrossmonthly__income_sum',
        'fltgrossmonthly__income_count',
    ]
    # convert str_values to a df
    X = pd.read_csv(
        StringIO(str_values), 
        delimiter=',', 
    )
    # if X is empty
    if X.empty:
        # append error
        dict_list_errors[unique_id].append('Missing Income Table')
        X = pd.DataFrame({col: np.nan for col in list_cols_agg}, index=[0])
    else:
        # lower
        X.columns = [col.lower() for col in X.columns]
        # rm T/F
        X_tmp = X * 1
        # filter rows
        X_tmp = X_tmp[X_tmp['bitinvalid']==0] # False
        X_tmp = X_tmp[X_tmp['bituse']==1] # True
        # aggregate
        X = pd.DataFrame()
        X['fltgrossmonthly__income_sum'] = [np.sum(X_tmp['fltgrossmonthly'])]
        X['fltgrossmonthly__income_count'] = [X_tmp.shape[0]]
    # set index to unique id
    X.index = [unique_id]
    # return
    return X, dict_list_errors

# get ln table
def get_lexis_nexis_table(str_values, unique_id, dict_list_errors):
    # read str_values to a df
    X = pd.read_csv(
        StringIO(str_values), 
        delimiter=',', 
    )
    # if empty
    if X.empty:
        # append error
        dict_list_errors[unique_id].append('Missing Lexis Nexis Table')
        # create row with NaN
        list_cols = [
            'ln_was_empty',
        ]
        X = pd.DataFrame({col: np.nan for col in list_cols}, index=[0])
    else:
        # lower
        X.columns = [col.lower() for col in X.columns]
    # add suffix to end of column names
    X.columns = [f'{col}__ln' for col in X.columns]
    # set index to unique id
    X.index = [unique_id]
    # return
    return X, dict_list_errors

# define helper to convert to proper dtype
def helper_convert_dtype(str_col_value):
    # try converting to integer
    try:
        str_col_value = int(str_col_value)
    except:
        # try converting to float
        try:
            str_col_value = float(str_col_value)
        # leave as string
        except:
            pass
    # return
    return str_col_value

# define helper to parse tuxml
def helper_parse_tuxml(str_values):
    # get root
    root = ET.fromstring(str_values)
    # empty dict
    dict_tuxml = {}
    # iterate through child branches
    for child in root.iter(tag='{http://www.transunion.com/namespace}characteristic'):
        # get col name
        str_col_name = child.find('{http://www.transunion.com/namespace}id').text.lower()
        # get col val
        try:
            str_col_value = helper_convert_dtype(str_col_value=child.find('{http://www.transunion.com/namespace}value').text)
        # if its nonetype
        except AttributeError:
            str_col_value = np.nan
        # assign
        dict_tuxml[str_col_name] = str_col_value
    # convert dtype and return df
    return pd.DataFrame(dict_tuxml, index=[0])

# get tu table
def get_transunion_table(str_values, unique_id, dict_list_errors):
    # parse xml
    X = helper_parse_tuxml(
        str_values=str_values, 
    )
    # if empty
    if X.empty:
        # append error
        dict_list_errors[unique_id].append('Missing TU Table')
        list_cols = [
            'tu_was_empty',
        ]
        # create row with NaN
        X = pd.DataFrame({col: np.nan for col in list_cols}, index=[0])
    else:
        pass
    # add suffix to end of column names
    X.columns = [f'{col}__tu' for col in X.columns]
    # set index to unique id
    X.index = [unique_id]
    # return
    return X, dict_list_errors

Overwriting functions.py


#### Write api.py

In [5]:
%%writefile api.py

from functions import *
import time

# define class
class ParsePayload:
    # init
    def __init__(self, list_cols_raw_all):
        self.str_datecol = 'applicationdate'
        self.dict_output = {}
        self.list_cols_raw_all = list_cols_raw_all
    # get data
    def get_data(self, dict_json_request):
        # start time
        time_start = time.perf_counter()

        # use helper
        list_unique_id, dict_list_tables = get_data_helper(
            dict_json_request=dict_json_request,
        )

        # save to object now for uniformity later
        self.dict_output['list_unique_id'] = list_unique_id

        # flt_sec
        flt_sec = time.perf_counter() - time_start
        # print
        print(f'{self.dict_output["list_unique_id"]}: Get Data: {flt_sec:0.5f} sec.')

        # save to object
        self.dict_json_request = dict_json_request
        # save to dict_output
        self.dict_output['flt_sec_get_data'] = flt_sec
        self.dict_output['dict_list_tables'] = dict_list_tables
        # return object
        return self
    # parse data
    def parse_data(self):
        # start time
        time_start = time.perf_counter()

        # empty dict
        dict_list_x = {}
        # another empty dict
        dict_list_errors = {}
        # iterate through unique ids
        for unique_id in self.dict_output['list_unique_id']:
            # put empty list in dict errors
            dict_list_errors[unique_id] = []
            # empty list
            dict_list_x[unique_id] = []
            # get tables
            list_dict_tables = self.dict_output['dict_list_tables'][unique_id]
            # iterate through tables
            for dict_table in list_dict_tables:
                # get str_values
                str_values = dict_table['values']
                # Application table
                if dict_table['name'] == 'Application':
                    # get application table
                    X, dict_list_errors = get_application_table(
                        str_values=str_values, 
                        unique_id=unique_id, 
                        dict_list_errors=dict_list_errors,
                    )
                    dict_list_x[unique_id].append(X)
                # Income table
                elif dict_table['name'] == 'Incomes':
                    # get income table
                    X, dict_list_errors = get_income_table(
                        str_values=str_values, 
                        unique_id=unique_id, 
                        dict_list_errors=dict_list_errors,
                    )
                    dict_list_x[unique_id].append(X)
                # LexisNexis
                elif dict_table['name'] == 'Lexis Nexis Risk View 5':
                    # get ln table
                    X, dict_list_errors = get_lexis_nexis_table(
                        str_values=str_values,
                        unique_id=unique_id, 
                        dict_list_errors=dict_list_errors,
                    )
                    dict_list_x[unique_id].append(X)
                # TU
                elif dict_table['name'] == 'TUXML':
                    # get tu table
                    X, dict_list_errors = get_transunion_table(
                        str_values=str_values,
                        unique_id=unique_id, 
                        dict_list_errors=dict_list_errors,
                    )
                    dict_list_x[unique_id].append(X)
                else:
                    pass

        # flt_sec
        flt_sec = time.perf_counter() - time_start
        # print
        print(f'{self.dict_output["list_unique_id"]}: Parse Data: {flt_sec:0.5f} sec.')

        # save to dict_output
        self.dict_output['flt_sec_parse_data'] = flt_sec
        self.dict_output['dict_list_x'] = dict_list_x
        self.dict_output['dict_list_errors'] = dict_list_errors
        # return object
        return self
    # create X
    def create_x(self):
        # start time
        time_start = time.perf_counter()

        # empty list
        list_x_cbind = []
        # concatenate cols of df in each list
        for list_x in self.dict_output['dict_list_x'].values():
            # concatenate
            x_cbind = pd.concat(list_x, axis=1)
            # append
            list_x_cbind.append(x_cbind)
        # concatenate rows
        X = pd.concat(list_x_cbind, axis=0)
        
        # ensure there is a field for every feature
        for col in self.list_cols_raw_all:
            if col not in list(X.columns):
                X[col] = np.nan

        # flt_sec
        flt_sec = time.perf_counter() - time_start
        # print()
        print(f'{self.dict_output["list_unique_id"]}: Create X: {flt_sec:0.5f} sec.')

        # save to dict_output
        self.dict_output['flt_sec_create_x'] = flt_sec
        self.dict_output['X_raw'] = X
        # return object
        return self

Overwriting api.py


#### Get features from raw data

In [6]:
%%time

str_filename = 'df_test_raw.gzip'
str_uri = f's3://20231010-gen-xii/03_pricing_lgd/01_data_prep/03_train_valid_test_split/{str_filename}'
list_cols_raw_all = list(pd.read_parquet(str_uri).columns)
# rm base
list_cols_raw_all = [col for col in list_cols_raw_all if '__base' not in col]
# income
list_cols_ignore = [
    'fltgrossmonthly__income_min',
    'fltgrossmonthly__income_max',
    'fltgrossmonthly__income_median',
    'fltgrossmonthly__income_mean',
    'fltgrossmonthly__income_std',
]
list_cols_raw_all = [col for col in list_cols_raw_all if col not in list_cols_ignore]
print(f'There are {len(list_cols_raw_all)} total raw features')

There are 2466 total raw features
CPU times: user 604 ms, sys: 733 ms, total: 1.34 s
Wall time: 3.24 s


#### Initialize class

In [7]:
from api import ParsePayload

cls_parser = ParsePayload(
    list_cols_raw_all=list_cols_raw_all,
)

#### Save

In [8]:
str_filename = 'cls_parser.pkl'
str_local_path = f'{str_dirname_output}/{str_filename}'
pickle.dump(cls_parser, open(str_local_path, 'wb'))

#### Test to see if parser would behave the way we expect it to

In [9]:
df = pd.read_csv('./output/df_requests_subset.csv')
df

,bigAccountId,dtmFunded,strRequest
0,5714239,2021-07-26,"{""request_id"":""588761"",""rows"":[{""row_id"":""5714..."
1,5709609,2021-07-27,"{""request_id"":""589163"",""rows"":[{""row_id"":""5709..."
2,5718633,2021-07-27,"{""request_id"":""589402"",""rows"":[{""row_id"":""5718..."
3,5704668,2021-07-27,"{""request_id"":""590811"",""rows"":[{""row_id"":""5704..."
4,5712452,2021-07-27,"{""request_id"":""592777"",""rows"":[{""row_id"":""5712..."
...,...,...,...
95,5716645,2021-08-02,"{""request_id"":""593192"",""rows"":[{""row_id"":""5716..."
96,5706152,2021-08-02,"{""request_id"":""593948"",""rows"":[{""row_id"":""5706..."
97,5713490,2021-08-02,"{""request_id"":""596309"",""rows"":[{""row_id"":""5713..."
98,5721262,2021-08-02,"{""request_id"":""593996"",""rows"":[{""row_id"":""5721..."


In [10]:
# keep only the most recent payload
df.drop_duplicates(
    subset=['bigAccountId'],
    keep='last',
    inplace=True,
)
# show
df

,bigAccountId,dtmFunded,strRequest
0,5714239,2021-07-26,"{""request_id"":""588761"",""rows"":[{""row_id"":""5714..."
1,5709609,2021-07-27,"{""request_id"":""589163"",""rows"":[{""row_id"":""5709..."
2,5718633,2021-07-27,"{""request_id"":""589402"",""rows"":[{""row_id"":""5718..."
3,5704668,2021-07-27,"{""request_id"":""590811"",""rows"":[{""row_id"":""5704..."
4,5712452,2021-07-27,"{""request_id"":""592777"",""rows"":[{""row_id"":""5712..."
...,...,...,...
95,5716645,2021-08-02,"{""request_id"":""593192"",""rows"":[{""row_id"":""5716..."
96,5706152,2021-08-02,"{""request_id"":""593948"",""rows"":[{""row_id"":""5706..."
97,5713490,2021-08-02,"{""request_id"":""596309"",""rows"":[{""row_id"":""5713..."
98,5721262,2021-08-02,"{""request_id"":""593996"",""rows"":[{""row_id"":""5721..."


In [11]:
# sort
df.sort_values(by='dtmFunded', ascending=True, inplace=True)

# show
df

,bigAccountId,dtmFunded,strRequest
0,5714239,2021-07-26,"{""request_id"":""588761"",""rows"":[{""row_id"":""5714..."
16,5711466,2021-07-27,"{""request_id"":""589516"",""rows"":[{""row_id"":""5711..."
15,5702106,2021-07-27,"{""request_id"":""589491"",""rows"":[{""row_id"":""5702..."
14,5708876,2021-07-27,"{""request_id"":""589591"",""rows"":[{""row_id"":""5708..."
13,5708693,2021-07-27,"{""request_id"":""589143"",""rows"":[{""row_id"":""5708..."
...,...,...,...
94,5720091,2021-08-02,"{""request_id"":""592645"",""rows"":[{""row_id"":""5720..."
95,5716645,2021-08-02,"{""request_id"":""593192"",""rows"":[{""row_id"":""5716..."
96,5706152,2021-08-02,"{""request_id"":""593948"",""rows"":[{""row_id"":""5706..."
97,5713490,2021-08-02,"{""request_id"":""596309"",""rows"":[{""row_id"":""5713..."


In [12]:
# get min and max dates
dtm_min = df['dtmFunded'].min()
dtm_max = df['dtmFunded'].max()
print(f'Min date: {dtm_min}; Max date: {dtm_max}')

Min date: 2021-07-26; Max date: 2021-08-02


In [13]:
# get nrows
int_nrows = df.shape[0]

# divide by int_n_requests_per_lambda
int_n_lambdas = math.ceil(int_nrows / int_n_requests_per_lambda)
print(f'There will be {int_n_lambdas} lambdas')

There will be 4 lambdas


In [14]:
# create list to assign as new column
list_rows = list(np.tile(np.arange(1, int_n_lambdas+1), int_n_requests_per_lambda))
print(f'Length: {len(list_rows)}')
# make sure its the same length as df
list_rows = list_rows[:int_nrows]
print(f'Length: {len(list_rows)}')
# assign
df['rows'] = list_rows
# show
df

Length: 120
Length: 100


,bigAccountId,dtmFunded,strRequest,rows
0,5714239,2021-07-26,"{""request_id"":""588761"",""rows"":[{""row_id"":""5714...",1
16,5711466,2021-07-27,"{""request_id"":""589516"",""rows"":[{""row_id"":""5711...",2
15,5702106,2021-07-27,"{""request_id"":""589491"",""rows"":[{""row_id"":""5702...",3
14,5708876,2021-07-27,"{""request_id"":""589591"",""rows"":[{""row_id"":""5708...",4
13,5708693,2021-07-27,"{""request_id"":""589143"",""rows"":[{""row_id"":""5708...",1
...,...,...,...,...
94,5720091,2021-08-02,"{""request_id"":""592645"",""rows"":[{""row_id"":""5720...",4
95,5716645,2021-08-02,"{""request_id"":""593192"",""rows"":[{""row_id"":""5716...",1
96,5706152,2021-08-02,"{""request_id"":""593948"",""rows"":[{""row_id"":""5706...",2
97,5713490,2021-08-02,"{""request_id"":""596309"",""rows"":[{""row_id"":""5713...",3


In [15]:
df_ = df.iloc[0].strRequest
df_

'{"request_id":"588761","rows":[{"row_id":"5714239__7176826__20210720","sources":[{"name":"Application","version":1.0,"format":"csv","values":"UniqueID,bigAccountId,bigDebtorId,bitDebtor,strCity,strName,strZipCode,ApplicationDate,bitApproved,bitSystemDecline,ApprovalDate,bitFunded,FundedDate,dtmStampCreation,dtmApproved,dtmDeclined,dtmFunded,defaultDate,ChargeOffDate,defaultAmount,chargeoffamount,ApplicationMonth,ApplicationQuarter,ApplicationDayofWeek,bigDealerID,bitRolled,bigDealerTypeId,DealerState,bitLHMGroup,DealerCity,DealerZip,bitDealerApplicantSameState,bitDealerApplicantSameCity,bitDealerApplicantSameZip,strDealershipTrackerType,intType,fltAcquisitionFee,fltAddFee,fltAllowance,fltAmountFinanced,fltApprovedAPR_contract,fltApprovedDebtToIncome,fltApprovedDownTotal,fltApprovedLoanToValue,fltApprovedPayment,fltApprovedPriceWholesale,fltApprovedServiceContract,fltDocumentFee,fltDownCash,fltGapInsurance,fltInsuredDisabilityAmount,fltInsuredDisabilityPremium,fltInsuredLifeAmount,fltI

#### df_idx.csv - not utilized here

In [16]:
list_rows = list(df['rows'].value_counts().index)
df_idx = pd.DataFrame({'row': list_rows})
df_idx.sort_values(by='row', ascending=True, inplace=True)

# save
str_filename = 'df_idx.csv'
df_idx.to_csv(f'{str_dirname_output}/{str_filename}', index=False)

# show
df_idx

,row
0,1
1,2
2,3
3,4


In [17]:
# parse (get income, LN, and TU)
print('Parsing requests...')
list_X_raw = []
for a, str_request in enumerate(df['strRequest']):
    # get bigAccountId
    int_bigaccountid = df['bigAccountId'].iloc[a]
    # get dtmFunded
    dtm_funded = df['dtmFunded'].iloc[a]
    
    # convert string request to dict
    dict_json_request = json.loads(str_request)
    # get data
    cls_parser.get_data(dict_json_request)
    # parse data
    cls_parser.parse_data()
    # create X
    cls_parser.create_x()
    # get X_raw
    X_raw = cls_parser.dict_output['X_raw']
        
    # assign
    X_raw['bigAccountId'] = int_bigaccountid
    X_raw['dtmFunded'] = dtm_funded
    # get nrows
    int_nrows = X_raw.shape[0]
    if int_nrows == 1:
        X_raw['BITDEBTOR'] = 1
    else:
        X_raw['BITDEBTOR'] = [1,0]
        
    # append
    list_X_raw.append(X_raw)
    
# concat
print('Concatenating raw data...')
X_raw = pd.concat(list_X_raw)
    
# save memeory
del list_X_raw
    
# write to s3 as parquet
print('Writing raw data to s3...')
# set nonnumeric to string
for col in X_raw.columns:
    # if not numeric
    if X_raw[col].dtype not in ['int64', 'float64']:
        # set as string
        X_raw[col] = X_raw[col].astype(str)
    else:
        pass

Parsing requests...
['5714239__7176826__20210720']: Get Data: 0.00000 sec.
['5714239__7176826__20210720']: Parse Data: 0.03457 sec.
['5714239__7176826__20210720']: Create X: 0.55109 sec.
['5711466__7173427__20210717']: Get Data: 0.00000 sec.
['5711466__7173427__20210717']: Parse Data: 0.03036 sec.
['5711466__7173427__20210717']: Create X: 0.59362 sec.
['5702106__7162085__20210707']: Get Data: 0.00000 sec.
['5702106__7162085__20210707']: Parse Data: 0.02926 sec.
['5702106__7162085__20210707']: Create X: 0.58711 sec.
['5708876__7170291__20210714']: Get Data: 0.00000 sec.
['5708876__7170291__20210714']: Parse Data: 0.03349 sec.
['5708876__7170291__20210714']: Create X: 0.51120 sec.
['5708693__7170066__20210714']: Get Data: 0.00000 sec.
['5708693__7170066__20210714']: Parse Data: 0.02927 sec.
['5708693__7170066__20210714']: Create X: 0.58711 sec.
['5712734__7174995__20210719']: Get Data: 0.00000 sec.
['5712734__7174995__20210719']: Parse Data: 0.03282 sec.
['5712734__7174995__20210719']: C

['5717785__7181140__20210724']: Create X: 0.50989 sec.
['5713040__7175367__20210719']: Get Data: 0.00000 sec.
['5713040__7175367__20210719']: Parse Data: 0.03272 sec.
['5713040__7175367__20210719']: Create X: 0.51097 sec.
['5714847__7177553__20210721']: Get Data: 0.00000 sec.
['5714847__7177553__20210721']: Parse Data: 0.03009 sec.
['5714847__7177553__20210721']: Create X: 0.58505 sec.
['5712088__7174212__20210717']: Get Data: 0.00000 sec.
['5712088__7174212__20210717']: Parse Data: 0.03328 sec.
['5712088__7174212__20210717']: Create X: 0.50914 sec.
['5715974__7178946__20210722', '5715974__7178947__20210722']: Get Data: 0.00000 sec.
['5715974__7178946__20210722', '5715974__7178947__20210722']: Parse Data: 0.12748 sec.
['5715974__7178946__20210722', '5715974__7178947__20210722']: Create X: 0.60624 sec.
['5708421__7169743__20210714', '5708421__7169744__20210714']: Get Data: 0.00000 sec.
['5708421__7169743__20210714', '5708421__7169744__20210714']: Parse Data: 0.06540 sec.
['5708421__7169

['5715012__7177763__20210721']: Create X: 0.51608 sec.
['5716145__7179144__20210722']: Get Data: 0.00000 sec.
['5716145__7179144__20210722']: Parse Data: 0.03404 sec.
['5716145__7179144__20210722']: Create X: 0.51170 sec.
['5721262__7185423__20210728']: Get Data: 0.00000 sec.
['5721262__7185423__20210728']: Parse Data: 0.03437 sec.
['5721262__7185423__20210728']: Create X: 0.51254 sec.
['5715716__7178627__20210722', '5715716__7178628__20210722']: Get Data: 0.00000 sec.
['5715716__7178627__20210722', '5715716__7178628__20210722']: Parse Data: 0.06520 sec.
['5715716__7178627__20210722', '5715716__7178628__20210722']: Create X: 0.53586 sec.
['5720007__7183907__20210727', '5720007__7183908__20210727']: Get Data: 0.00000 sec.
['5720007__7183907__20210727', '5720007__7183908__20210727']: Parse Data: 0.05931 sec.
['5720007__7183907__20210727', '5720007__7183908__20210727']: Create X: 0.60662 sec.
['5720091__7184009__20210727']: Get Data: 0.00000 sec.
['5720091__7184009__20210727']: Parse Data

In [18]:
X_raw

,uniqueid__app,bigaccountid__app,bigdebtorid__app,bitdebtor__app,strcity__app,strname__app,strzipcode__app,applicationdate__app,bitapproved__app,bitsystemdecline__app,...,fltadvance__app,strvehicletype__app,bitgap__app,dealerstampcreation__app,RunningNetLoss,bitTarget24Months,target,bigAccountId,dtmFunded,BITDEBTOR
5714239__7176826__20210720,5714239__7176826__20210720,5714239,7176826,1,W VALLEY CITY,Utah,84119,NaN,True,nan,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5714239,2021-07-26,1
5711466__7173427__20210717,5711466__7173427__20210717,5711466,7173427,1,BRIERFIELD,Alabama,35035,NaN,True,nan,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5711466,2021-07-27,1
5702106__7162085__20210707,5702106__7162085__20210707,5702106,7162085,1,SACRAMENTO,California,95823,NaN,True,nan,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5702106,2021-07-27,1
5708876__7170291__20210714,5708876__7170291__20210714,5708876,7170291,1,MCFARLAND,Wisconsin,53558,NaN,True,nan,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5708876,2021-07-27,1
5708693__7170066__20210714,5708693__7170066__20210714,5708693,7170066,1,MATTESON,Illinois,60443,NaN,True,nan,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5708693,2021-07-27,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5716645__7179744__20210723,5716645__7179744__20210723,5716645,7179744,0,Rock Falls,Illinois,61071,NaN,True,nan,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5716645,2021-08-02,0
5706152__7166995__20210712,5706152__7166995__20210712,5706152,7166995,1,SAVAGE,Minnesota,55378,NaN,True,nan,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5706152,2021-08-02,1
5713490__7175919__20210720,5713490__7175919__20210720,5713490,7175919,1,MOORESVILLE,Indiana,46158,NaN,True,nan,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5713490,2021-08-02,1
5713619__7176075__20210720,5713619__7176075__20210720,5713619,7176075,1,COLORADO SPRINGS,Colorado,80920,NaN,True,nan,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5713619,2021-08-02,1


In [19]:
list_ = list(X_raw.columns)
list_

['uniqueid__app',
 'bigaccountid__app',
 'bigdebtorid__app',
 'bitdebtor__app',
 'strcity__app',
 'strname__app',
 'strzipcode__app',
 'applicationdate__app',
 'bitapproved__app',
 'bitsystemdecline__app',
 'approvaldate__app',
 'bitfunded__app',
 'fundeddate__app',
 'dtmstampcreation__app',
 'dtmapproved__app',
 'dtmdeclined__app',
 'dtmfunded__app',
 'defaultdate__app',
 'chargeoffdate__app',
 'defaultamount__app',
 'chargeoffamount__app',
 'applicationmonth__app',
 'applicationquarter__app',
 'applicationdayofweek__app',
 'bigdealerid__app',
 'bitrolled__app',
 'bigdealertypeid__app',
 'dealerstate__app',
 'bitlhmgroup__app',
 'dealercity__app',
 'dealerzip__app',
 'bitdealerapplicantsamestate__app',
 'bitdealerapplicantsamecity__app',
 'bitdealerapplicantsamezip__app',
 'strdealershiptrackertype__app',
 'inttype__app',
 'fltacquisitionfee__app',
 'fltaddfee__app',
 'fltallowance__app',
 'fltamountfinanced__app',
 'fltapprovedapr_contract__app',
 'fltapproveddebttoincome__app',
 'fl

In [20]:
list_tu = [col for col in X_raw.columns if '__tu' in col.lower()]
X_raw[list_tu]

,at01s__tu,at02s__tu,at03s__tu,at06s__tu,at09s__tu,at12s__tu,at20s__tu,at21s__tu,at24s__tu,at25s__tu,...,us934c__tu,us934d__tu,us934s__tu,us935b__tu,us935c__tu,us935d__tu,us935s__tu,score_bankcard__tu,score_cvpropensity__tu,finscore__tu
5714239__7176826__20210720,3,1,1,1,2,1,112,2,-6,-6,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5711466__7173427__20210717,5,0,-3,0,0,-3,122,62,-3,-3,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5702106__7162085__20210707,2,0,-3,0,0,-3,189,181,-3,-3,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5708876__7170291__20210714,13,1,1,0,0,1,48,32,1,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5708693__7170066__20210714,3,2,2,0,2,2,136,14,2,2,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5716645__7179744__20210723,8,0,-3,0,2,-3,79,8,-3,-3,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5706152__7166995__20210712,11,0,-3,0,0,-3,223,56,-3,-3,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5713490__7175919__20210720,7,1,1,2,4,1,74,4,-6,-6,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5713619__7176075__20210720,26,0,-3,0,2,-3,150,17,-3,-3,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [21]:
list_ln = [col for col in X_raw.columns if '__ln' in col.lower()]
X_raw[list_ln]

,uniqueid__ln,biglnriskviewattributesv5id__ln,bigaccountid__ln,bigdebtorid__ln,biglnriskviewscoreid__ln,bitinvalid__ln,dtmstampcreation__ln,attribute_index__ln,inputprovidedfirstname__ln,inputprovidedlastname__ln,...,alert2__ln,alert3__ln,alert4__ln,alert5__ln,alert6__ln,alert7__ln,alert8__ln,alert9__ln,alert10__ln,intscore__ln
5714239__7176826__20210720,"5714239,7176826",0,5714239,7176826,0,False,2021-07-20T17:06:40,NaN,1.0,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5711466__7173427__20210717,"5711466,7173427",0,5711466,7173427,0,False,2021-07-17T11:00:51,NaN,1.0,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5702106__7162085__20210707,"5702106,7162085",0,5702106,7162085,0,False,2021-07-07T12:17:57,NaN,1.0,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5708876__7170291__20210714,"5708876,7170291",0,5708876,7170291,0,False,2021-07-14T17:50:30,NaN,1.0,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5708693__7170066__20210714,"5708693,7170066",0,5708693,7170066,0,False,2021-07-14T15:56:05,NaN,1.0,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5716645__7179744__20210723,"5716645,7179744",0,5716645,7179744,0,False,2021-07-23T11:42:57,NaN,1.0,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5706152__7166995__20210712,"5706152,7166995",0,5706152,7166995,0,False,2021-07-12T09:24:55,NaN,1.0,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5713490__7175919__20210720,"5713490,7175919",0,5713490,7175919,0,False,2021-07-20T08:34:21,NaN,1.0,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5713619__7176075__20210720,"5713619,7176075",0,5713619,7176075,0,False,2021-07-20T10:22:42,NaN,1.0,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
